In [17]:
"""
Cleaning / preprocessing for the combined London OpenActive dataset.
Each "# %%" marks a separate runnable cell in VS Code's Jupyter extension —
run them one at a time so you can see exactly where things break.
"""

# %% ================================================================
# BLOCK 1 — LOAD + INSPECT
# Just loads the file and shows you what you're actually working with
# before any cleaning happens. Always start here after any new file.
# =====================================================================
import pandas as pd
import numpy as np

INPUT_PATH = r"C:\Users\Hp\Downloads\Project 2026 DS\Person2_Combined_London.csv"
OUTPUT_PATH = r"C:\Users\Hp\Downloads\Project 2026 DS\Person2_Combined_London_clean.csv"

df = pd.read_csv(INPUT_PATH)

print(f"Loaded: {len(df)} rows, {len(df.columns)} columns")
print("\nColumn names:", df.columns.tolist())
print("\nDtypes:\n", df.dtypes)
print("\nFirst 3 rows:\n", df.head(3))

print("Duplicate index labels:", df.index.duplicated().sum())
print(df.columns[df.columns.duplicated()])  # double-check column names too
# %% ================================================================
# BLOCK 2 — COLUMN NAME DIAGNOSTICS (run this if you see reindex /
# duplicate-label errors, or if this file came from a merge/concat)
# =====================================================================
print("Columns:", df.columns.tolist())

dupe_cols = df.columns[df.columns.duplicated()]
print("Duplicate column names:", dupe_cols.tolist())

if len(dupe_cols) > 0:
    # Keeps the FIRST occurrence of each duplicated name, drops the rest.
    # Only safe if you've checked the dropped columns aren't the ones
    # you actually want — print df[dupe_cols] first if unsure.
    df = df.loc[:, ~df.columns.duplicated()]
    print(f"Dropped duplicate-named columns. New shape: {df.shape}")

# If instead you see lat_x / lat_y / lon_x / lon_y (typical after a
# pd.merge() where both sides had lat/lon) — uncomment and adapt:
#
# df["lat"] = df["lat_x"].fillna(df["lat_y"])
# df["lon"] = df["lon_x"].fillna(df["lon_y"])
# df = df.drop(columns=["lat_x", "lat_y", "lon_x", "lon_y"])

# Standardise names (lowercase, underscores) once columns are sane
df.columns = [c.strip().lower().replace(" ", "_") for c in df.columns]
print("\nFinal column names:", df.columns.tolist())


# %% ================================================================
# BLOCK 2.5 — DROP LEFTOVER SPATIAL JOIN COLUMNS
# These came along from gpd.sjoin() against the boundary file and
# aren't needed downstream. LAD24CD/LAD24NM are the useful borough
# identifiers — keep those, drop the rest.
# =====================================================================
sjoin_junk = ["geometry", "index_right", "fid", "lad24nmw",
              "bng_e", "bng_n", "long", "lat", "globalid"]

# careful: this drops the SECOND "lat" (the boundary file's own LAT
# column, post-lowercasing) — confirm which one is which before running
print(df[[c for c in sjoin_junk if c in df.columns]].head())

df = df.loc[:, ~df.columns.duplicated()]  # keep first "lat" (your point data)
df = df.drop(columns=[c for c in sjoin_junk if c in df.columns and c not in ("lat", "long")])

print("Columns after cleanup:", df.columns.tolist())



Loaded: 42547 rows, 20 columns

Column names: ['item_id', 'venue_name', 'facility_name', 'lat', 'lon', 'start_date', 'geometry', 'index_right', 'FID', 'LAD24CD', 'LAD24NM', 'LAD24NMW', 'BNG_E', 'BNG_N', 'LONG', 'LAT', 'GlobalID', 'provider', 'feed_type', 'activity']

Dtypes:
 item_id           object
venue_name        object
facility_name     object
lat              float64
lon              float64
start_date        object
geometry          object
index_right      float64
FID              float64
LAD24CD           object
LAD24NM           object
LAD24NMW          object
BNG_E            float64
BNG_N            float64
LONG             float64
LAT              float64
GlobalID          object
provider          object
feed_type         object
activity          object
dtype: object

First 3 rows:
      item_id             venue_name              facility_name        lat  \
0  140068060  Putney Leisure Centre  Good Boost Grab & Go Aqua  51.464044   
1  140068477  Putney Leisure Centre  Go

C:\Users\Hp\AppData\Local\Temp\ipykernel_90760\2206692800.py:18: DtypeWarning: Columns (19) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(INPUT_PATH)


In [18]:


# %% ================================================================
# BLOCK 3 — DEDUPLICATION
# Exact duplicate rows, then duplicate item_id (the real uniqueness key)
# =====================================================================
before = len(df)
df = df.drop_duplicates()
print(f"Exact duplicate rows dropped: {before - len(df)} (now {len(df)})")

if "item_id" in df.columns:
    dupe_ids = df["item_id"].duplicated().sum()
    print(f"Rows with duplicate item_id: {dupe_ids}")
    df = df.drop_duplicates(subset="item_id", keep="first")
    print(f"After item_id dedup: {len(df)} rows")


# %% ================================================================
# BLOCK 4 — MISSING / INVALID COORDINATES
# Missing geo is a genuine finding, not junk — split it off rather
# than dropping it silently. Also sanity-checks lat/lon are in the UK.
# =====================================================================
if {"lat", "lon"}.issubset(df.columns):
    missing_geo = df[df["lat"].isna() | df["lon"].isna()].copy()
    df_geo = df.dropna(subset=["lat", "lon"]).copy()

    print(f"Rows with missing geo: {len(missing_geo)} ({len(missing_geo)/len(df):.1%})")

    missing_geo.to_csv(
        r"C:\Users\Hp\Downloads\Project 2026 DS\Person2_missing_geo.csv", index=False
    )

    # UK bounding box sanity check — catches swapped lat/lon or bad geocoding
    bad_coords = df_geo[
        ~df_geo["lat"].between(49, 61) | ~df_geo["lon"].between(-8, 2)
    ]
    print(f"Rows with out-of-range UK coordinates: {len(bad_coords)}")
    df_geo = df_geo[~df_geo.index.isin(bad_coords.index)]
else:
    print("No lat/lon columns found — skipping geo checks.")
    df_geo = df.copy()

print(f"Remaining rows after geo checks: {len(df_geo)}")



Exact duplicate rows dropped: 12 (now 42535)
Rows with duplicate item_id: 0
After item_id dedup: 42535 rows
Rows with missing geo: 0 (0.0%)
Rows with out-of-range UK coordinates: 0
Remaining rows after geo checks: 42535


In [19]:

# %% ================================================================
# BLOCK 5 — DATES
# Parses start_date properly, flags anything unparseable instead of
# silently turning it into NaT, and adds a London-local time column.
# =====================================================================
if "start_date" in df_geo.columns:
    parsed = pd.to_datetime(df_geo["start_date"], errors="coerce", utc=True)
    bad_dates = parsed.isna().sum()
    print(f"Unparseable start_date values: {bad_dates}")

    df_geo["start_date_parsed"] = parsed
    df_geo["start_date_local"] = parsed.dt.tz_convert("Europe/London")
else:
    print("No start_date column found — skipping date parsing.")


# %% ================================================================
# BLOCK 6 — TEXT CLEANUP
# Strips whitespace and collapses repeated spaces in name/category
# fields. Fuzzy venue-name matching is included but off by default
# (slow on 40k+ rows — only turn on if you spot a lot of near-dupes).
# =====================================================================
text_cols = [c for c in ["venue_name", "activity_type", "provider"] if c in df_geo.columns]

for col in text_cols:
    df_geo[col] = (
        df_geo[col]
        .astype(str)
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )

print(f"Cleaned text columns: {text_cols}")

# Optional fuzzy dedup of venue names — uncomment if needed
# (pip/conda install rapidfuzz first)
#
# from rapidfuzz import fuzz, process
# def fuzzy_canonicalise(series, threshold=90):
#     unique_vals = series.dropna().unique()
#     mapping = {}
#     for val in unique_vals:
#         if val in mapping:
#             continue
#         matches = process.extract(val, unique_vals, scorer=fuzz.ratio, limit=None)
#         for match_val, score, _ in matches:
#             if score >= threshold:
#                 mapping[match_val] = val
#     return series.map(mapping).fillna(series)
#
# if "venue_name" in df_geo.columns:
#     df_geo["venue_name"] = fuzzy_canonicalise(df_geo["venue_name"])


# %% ================================================================
# BLOCK 7 — BOROUGH CLEANUP
# Tidies the borough column from the spatial join and reports coverage
# =====================================================================
borough_col = next((c for c in df_geo.columns if "borough" in c or "lad24nm" in c), None)

if borough_col:
    df_geo[borough_col] = df_geo[borough_col].astype(str).str.strip()
    n_boroughs = df_geo[borough_col].nunique()
    print(f"Unique boroughs present: {n_boroughs} / 33")
    print("Boroughs covered:", sorted(df_geo[borough_col].unique()))
else:
    print("No borough column found — skipping.")


# %% ================================================================
# BLOCK 8 — FINAL SUMMARY + SAVE
# =====================================================================
print("\n--- Summary ---")
print(f"Final clean rows: {len(df_geo)}")
print(f"Columns: {list(df_geo.columns)}")

if "provider" in df_geo.columns:
    print(df_geo["provider"].value_counts())

df_geo.to_csv(OUTPUT_PATH, index=False)
print(f"\nSaved cleaned file to: {OUTPUT_PATH}")

Unparseable start_date values: 5484
Cleaned text columns: ['venue_name', 'provider']
Unique boroughs present: 12 / 33
Boroughs covered: ['Barking and Dagenham', 'Barnet', 'Brent', 'Ealing', 'Harrow', 'Havering', 'Kensington and Chelsea', 'Kingston upon Thames', 'Newham', 'Sutton', 'Wandsworth', 'Westminster']

--- Summary ---
Final clean rows: 42535
Columns: ['item_id', 'venue_name', 'facility_name', 'lat', 'lon', 'start_date', 'lad24cd', 'lad24nm', 'long', 'provider', 'feed_type', 'activity', 'start_date_parsed', 'start_date_local']
provider
Everyone Active    25767
Places Leisure     16768
Name: count, dtype: int64

Saved cleaned file to: C:\Users\Hp\Downloads\Project 2026 DS\Person2_Combined_London_clean.csv


In [23]:
# %% ================================================================
# BLOCK 9 — AGGREGATE TO VENUE/ACTIVITY LEVEL
# Collapses individual booking-slot rows into one row per unique
# (venue, facility_name, provider) combination — the right granularity
# for "what's available where" rather than "how many time slots".
# Uses facility_name (not the old "activity" column, which was
# returning "Slot" for Places Leisure rows pre-fix).
# =====================================================================

group_cols = ["venue_name", "facility_name", "provider"]
if "lad24nm" in df_geo.columns:
    group_cols.append("lad24nm")  # keep borough alongside, since it's 1:1 with venue

agg_dict = {
    "item_id": "count",          # how many individual sessions roll up into this row
    "lat": "first",
    "lon": "first",
}
if "start_date_parsed" in df_geo.columns:
    agg_dict["start_date_parsed"] = ["min", "max"]  # date range this activity runs over

df_agg = df_geo.groupby(group_cols, dropna=False).agg(agg_dict)

# Flatten the multi-level columns created by the list-valued agg above
df_agg.columns = ["_".join(c).strip("_") if isinstance(c, tuple) else c for c in df_agg.columns]
df_agg = df_agg.rename(columns={"item_id_count": "n_sessions"}).reset_index()

print(f"Collapsed {len(df_geo)} booking-slot rows -> {len(df_agg)} unique venue/facility combinations")
print(df_agg.head())

df_agg.to_csv(
    r"C:\Users\Hp\Downloads\Project 2026 DS\Person2_VenueActivity_London.csv", index=False
)

Collapsed 42535 booking-slot rows -> 14704 unique venue/facility combinations
                  venue_name facility_name         provider   lad24nm  \
0  1-2-1 Add Needs Mon 16:00           NaN  Everyone Active  Havering   
1  1-2-1 Add Needs Mon 16:30           NaN  Everyone Active  Havering   
2  1-2-1 Add Needs Mon 17:30           NaN  Everyone Active  Havering   
3  1-2-1 Add Needs Mon 18:00           NaN  Everyone Active  Havering   
4  1-2-1 Add Needs Mon 18:30           NaN  Everyone Active  Havering   

   n_sessions  lat_first  lon_first start_date_parsed_min  \
0           1   51.57715   0.185115                   NaT   
1           1   51.57715   0.185115                   NaT   
2           1   51.57715   0.185115                   NaT   
3           1   51.57715   0.185115                   NaT   
4           1   51.57715   0.185115                   NaT   

  start_date_parsed_max  
0                   NaT  
1                   NaT  
2                   NaT  
3           

In [21]:
import pandas as pd
df_check = pd.read_csv(r"C:\Users\Hp\Downloads\Project 2026 DS\Person2_Combined_London.csv")
print(df_check.columns.tolist())
print(df_check["venue_name"].value_counts().head(10))
print(len(df_check))

['item_id', 'venue_name', 'facility_name', 'lat', 'lon', 'start_date', 'geometry', 'index_right', 'FID', 'LAD24CD', 'LAD24NM', 'LAD24NMW', 'BNG_E', 'BNG_N', 'LONG', 'LAT', 'GlobalID', 'provider', 'feed_type', 'activity']
venue_name
Latchmere Leisure Centre             3194
Tooting Leisure Centre               2601
Tolworth Recreation Centre           2029
Wandle Recreation Centre             2021
London Aquatics Centre               1971
Roehampton Sport & Fitness Centre    1911
Malden Centre                        1770
Balham Leisure Centre                1538
Putney Leisure Centre                1490
Sapphire Ice and Leisure             1222
Name: count, dtype: int64
42547


C:\Users\Hp\AppData\Local\Temp\ipykernel_90760\3210208212.py:2: DtypeWarning: Columns (19) have mixed types. Specify dtype option on import or set low_memory=False.
  df_check = pd.read_csv(r"C:\Users\Hp\Downloads\Project 2026 DS\Person2_Combined_London.csv")


In [22]:
import json

with open(r"C:\Users\Hp\Downloads\Project 2026 DS\Places_Leisure__FacilityUse.json") as f:
    pl_facilities = json.load(f)

for item in pl_facilities[:5]:
    d = item.get("data", {})
    print("ITEM ID:", d.get("@id"))
    print("top-level name:", d.get("name"))
    print("location field:", d.get("location"))
    print("---")

ITEM ID: https://placesleisure.gs-signature.cloud/OpenActive/api/facility-uses/086A200108
top-level name: Fun Float Splash Teach
location field: {'@type': 'Place', 'identifier': '86', 'name': 'Strode Leisure Centre', 'description': 'We have a range of pool activities at Strode Leisure Centre. If you wish to learn how to swim or want to enjoy a lane swim session, we have something for you!\n\nOur gym accommodates to a variety of abilities and goals, helping our members stay active however they like. If group workout classes are more your thing, we offer classes such as Body Conditioning, Aquafit and Group Cycling across all intensities.\xa0\n\nWe also work with National Governing Bodies to ensure our centre has fun, engaging sports. Get stuck into table tennis, or make a hit on one of our badminton courts!\xa0\n\nWe have a range of birthday party options available to help you celebrate, such as an Active Play and Bounce or Aqua Run Party.\n', 'address': {'@type': 'PostalAddress', 'addre

In [24]:
df_geo = pd.read_csv(r"C:\Users\Hp\Downloads\Project 2026 DS\Person2_Combined_London_clean.csv")

group_cols = ["venue_name", "facility_name", "provider", "lad24nm"]

agg_dict = {
    "item_id": "count",
    "lat": "first",
    "lon": "first",
}

df_agg = df_geo.groupby(group_cols, dropna=False).agg(agg_dict)
df_agg.columns = ["_".join(c).strip("_") if isinstance(c, tuple) else c for c in df_agg.columns]
df_agg = df_agg.rename(columns={"item_id_count": "n_sessions"}).reset_index()

print(f"Collapsed {len(df_geo)} rows -> {len(df_agg)} unique venue/facility combinations")
print(df_agg[df_agg["provider"].str.contains("Everyone", case=False)].head(10))

Collapsed 42535 rows -> 14704 unique venue/facility combinations
                  venue_name facility_name         provider  \
0  1-2-1 Add Needs Mon 16:00           NaN  Everyone Active   
1  1-2-1 Add Needs Mon 16:30           NaN  Everyone Active   
2  1-2-1 Add Needs Mon 17:30           NaN  Everyone Active   
3  1-2-1 Add Needs Mon 18:00           NaN  Everyone Active   
4  1-2-1 Add Needs Mon 18:30           NaN  Everyone Active   
5  1-2-1 Add Needs Sun 08:30           NaN  Everyone Active   
6         16-18 Fri 19:30 Ag           NaN  Everyone Active   
7  16:30 1-2-1 Inclusive Mon           NaN  Everyone Active   
8  16:30 121 Inclusive Thurs           NaN  Everyone Active   
9  17:00 1-2-1 Inclusive Mon           NaN  Everyone Active   

                lad24nm  item_id        lat       lon  
0              Havering        1  51.577150  0.185115  
1              Havering        1  51.577150  0.185115  
2              Havering        1  51.577150  0.185115  
3              Ha